# 13 — Secure, promote and deploy

Inspect release evidence, package locally and use configured services for real inventory and deployment from notebook cells.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/13-secure-promote-deploy/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/13-secure-promote-deploy/lesson.md) · [Exercise](../modules/13-secure-promote-deploy/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/13-secure-promote-deploy"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Inspect the local assurance boundary

The bundled SBOM is a teaching fixture with integrity metadata. Checking it demonstrates coverage/checksum verification; it is not a fresh vulnerability assessment of a deployment. Inspect the actual inventory and evidence before selecting a release.


In [ ]:
(
set -euo pipefail
python "$WORKSPACE/scripts/supply_chain.py" inventory --contract "$WORKSPACE/reference/waterbodies.cwl"
python "$WORKSPACE/scripts/supply_chain.py" check     --contract "$WORKSPACE/reference/expected/sbom/local-example/waterbodies.cwl"     --bundle "$WORKSPACE/reference/expected/sbom/local-example"
python -m pytest "$WORKSPACE/tests/supply_chain"
)


## Package and inspect locally with ORAS

ORAS is an external CLI, not the Python package of the same name. If installed, the cell creates a local OCI layout, reads its manifest and digest, restores the contract and compares bytes. It makes no registry writes. If ORAS is missing, the cell reports the skipped packaging activity; the instructor should provision it for this module.


In [ ]:
(
set -euo pipefail
if command -v oras >/dev/null 2>&1; then
    oras version
    transpiler-mate cwl2oci --output annotations.json "$WORKSPACE/reference/waterbodies.cwl#waterbodies"
    bash "$WORKSPACE/supply-chain/oras/package.sh" 1.0.0 "$WORKSPACE/reference/waterbodies.cwl" annotations.json "$LAB_DIR/oci"
    oras manifest fetch --oci-layout "$LAB_DIR/oci:1.0.0" > manifest.json
    oras manifest fetch --descriptor --oci-layout "$LAB_DIR/oci:1.0.0" > descriptor.json
    cat descriptor.json
    oras pull --oci-layout "$LAB_DIR/oci:1.0.0" -o restored
    cmp "$WORKSPACE/reference/waterbodies.cwl" restored/waterbodies.cwl
else
    printf 'OCI packaging skipped: the instructor must provision the ORAS CLI.\n'
fi
)


## Optional: generate fresh registry evidence and scan it

The instructor supplies `LAB_RELEASE_CONTRACT` pointing to the reviewed release contract whose four versioned images are available in an accessible registry. Trivy inspects remote images, even if Docker has local copies. Set `LAB_RUN_REGISTRY=1` to run generation, coverage/checksum checks and the HIGH/CRITICAL policy. A fresh output directory is allocated for every run. Do not proceed to promotion if a gate fails.


In [ ]:
export LAB_RUN_REGISTRY=${LAB_RUN_REGISTRY:-0}
export LAB_RELEASE_CONTRACT=${LAB_RELEASE_CONTRACT:-}


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_REGISTRY" == 1 ]]; then
    : "${LAB_RELEASE_CONTRACT:?Set the reviewed contract declaring accessible registry images}"
    test -r "$LAB_RELEASE_CONTRACT"
    trivy --version
    evidence_parent=$(mktemp -d "$LAB_DIR/evidence.XXXXXX")
    bundle="$evidence_parent/sbom"
    transpiler-mate cwl2sbom --platform linux/amd64 --output "$bundle" "$LAB_RELEASE_CONTRACT#waterbodies"
    python "$WORKSPACE/scripts/supply_chain.py" check --contract "$LAB_RELEASE_CONTRACT" --bundle "$bundle"
    bash "$WORKSPACE/supply-chain/policy/scan.sh" "$bundle"
    printf '%s\n' "$bundle" > evidence-location.txt
else
    printf 'Fresh registry evidence and vulnerability policy skipped: configure images and Trivy first.\n'
fi
)


## Optional: retrieve the reviewed promoted package

Promotion uses the reviewed release workflow and protected environment described in the [supply-chain guide](../docs/supply-chain.md). This lab does not publish or promote an unchecked candidate. Set `EOAP_REF` to the actual promoted `@sha256:` identity and `API` to your configured ZOO API. Authentication belongs in an instructor-provided curl config (`LAB_CURL_CONFIG`), never in notebook source or outputs.

Set `LAB_RUN_DEPLOYMENT=1` only after inspecting the promoted package and its attached evidence. The following cells contact that configured service and can install a process and submit a real job.


In [ ]:
export LAB_RUN_DEPLOYMENT=${LAB_RUN_DEPLOYMENT:-0}
export EOAP_REF=${EOAP_REF:-}
export API=${API:-}
export PROCESS_ID=${PROCESS_ID:-waterbodies}
export LAB_CURL_CONFIG=${LAB_CURL_CONFIG:-}
export LAB_EXECUTE_REQUEST=${LAB_EXECUTE_REQUEST:-}


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_DEPLOYMENT" == 1 ]]; then
    : "${EOAP_REF:?Set the reviewed promoted artifact digest}"
    [[ "$EOAP_REF" =~ @sha256:[[:xdigit:]]{64}$ ]]
    oras discover "$EOAP_REF"
    oras pull "$EOAP_REF" -o promoted
    cwltool --validate promoted/waterbodies.cwl#waterbodies
    python "$WORKSPACE/scripts/supply_chain.py" inventory --contract promoted/waterbodies.cwl
else
    printf 'Promoted package retrieval skipped: supply a reviewed digest first.\n'
fi
)


## Install, discover and describe the process

The server must be prepared by the course provider. These requests use the documented ZOO route structure and process selector. Read the returned identifier and update `PROCESS_ID` in the configuration cell if the server assigns another id. Generating an OGC process description does not install a process. Inspect `installed-package.json` in the editor and compare its scientific ports and image declarations with `promoted/waterbodies.cwl`; the completion record retains a checksum of the returned package representation.


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_DEPLOYMENT" == 1 ]]; then
    : "${API:?Set the actual ZOO API base URL}"
    curl_options=()
    if [[ -n "$LAB_CURL_CONFIG" ]]; then curl_options=(--config "$LAB_CURL_CONFIG"); fi
    curl "${curl_options[@]}" --fail-with-body -X POST "$API/processes?w=$PROCESS_ID"         -H 'Accept: application/json' -H 'Content-Type: application/cwl+yaml'         --data-binary @promoted/waterbodies.cwl > deployed-process.json
    cat deployed-process.json
    curl "${curl_options[@]}" --fail-with-body "$API/processes" > processes.json
    curl "${curl_options[@]}" --fail-with-body "$API/processes/$PROCESS_ID" > process.json
    curl "${curl_options[@]}" --fail-with-body "$API/processes/$PROCESS_ID/package" > installed-package.json
    cat process.json
else
    printf 'ZOO installation and discovery skipped: live deployment is disabled.\n'
fi
)


## Submit a backend-staged input

The backend must read the complete STAC Directory, including catalog, Item and band assets. A local workspace path or one Item URL is insufficient. The instructor supplies `LAB_EXECUTE_REQUEST` following the deployed process schema and verified backend staging convention; use [the deployment guide](../docs/deployment.md) to prepare it. This cell submits the real request and retains response headers and body.


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_DEPLOYMENT" == 1 ]]; then
    : "${LAB_EXECUTE_REQUEST:?Set the prepared backend-staged execution request file}"
    test -r "$LAB_EXECUTE_REQUEST"
    curl_options=()
    if [[ -n "$LAB_CURL_CONFIG" ]]; then curl_options=(--config "$LAB_CURL_CONFIG"); fi
    curl "${curl_options[@]}" --fail-with-body -D job.headers -o job.json         -X POST "$API/processes/$PROCESS_ID/execution"         -H 'Content-Type: application/json' -H 'Prefer: respond-async'         --data-binary "@$LAB_EXECUTE_REQUEST"
    cat job.json
else
    printf 'Remote job submission skipped: live deployment is disabled.\n'
fi
)


## Monitor and retrieve the real result

Read the returned job id in `job.json` or the Location header in `job.headers`, then set `JOB_ID` below. Rerun the status cell until the service reports successful completion. The results endpoint may not be ready before that. Set `LAB_RESULT_HREF` to the accessible STAC catalog or Item URL advertised by the result. The download cell below uses PySTAC to follow links and retrieves every asset into `LAB_RESULT_CATALOG`, preserving a portable product. It requires metadata URLs that PySTAC can read in this workspace; for a backend returning an archive or requiring a custom authenticated staging adapter, the instructor supplies that adapter as a notebook cell instead. The final validation checks the actual downloaded product.


In [ ]:
export JOB_ID=${JOB_ID:-}
export LAB_RESULT_HREF=${LAB_RESULT_HREF:-}
export LAB_RESULT_CATALOG=${LAB_RESULT_CATALOG:-$LAB_DIR/remote-result/catalog}


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_DEPLOYMENT" == 1 ]]; then
    : "${JOB_ID:?Set the returned job identifier}"
    curl_options=()
    if [[ -n "$LAB_CURL_CONFIG" ]]; then curl_options=(--config "$LAB_CURL_CONFIG"); fi
    curl "${curl_options[@]}" --fail-with-body "$API/jobs/$JOB_ID" > job-status.json
    cat job-status.json
    python - <<'PYTHON'
import json
from pathlib import Path
status = json.loads(Path('job-status.json').read_text())['status']
assert status == 'successful', f'Job is {status}; rerun this cell after inspecting server diagnostics'
PYTHON
    curl "${curl_options[@]}" --fail-with-body "$API/jobs/$JOB_ID/results" > job-results.json
    cat job-results.json
else
    printf 'Remote job monitoring skipped: live deployment is disabled.\n'
fi
)


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_DEPLOYMENT" == 1 ]]; then
    : "${LAB_RESULT_HREF:?Set the returned accessible STAC catalog or Item URL}"
    python - <<'PYTHON'
import os
import shutil
import subprocess
from pathlib import Path
from urllib.parse import unquote, urlparse
import pystac

remote = pystac.read_file(os.environ['LAB_RESULT_HREF'])
if isinstance(remote, pystac.Item):
    catalog = pystac.Catalog(id='downloaded', description='Retrieved processing result')
    catalog.add_item(remote.clone())
elif isinstance(remote, pystac.Catalog):
    catalog = remote.full_copy()
else:
    raise ValueError('The result must be a STAC Item or Catalog')
items = list(catalog.get_items(recursive=True))
assert items, 'The remote result has no Items'
for item in items:
    assert Path(item.id).name == item.id and item.id not in {'', '.', '..'}
asset_sources = {(item.id, key): asset.get_absolute_href()
                 for item in items for key, asset in item.assets.items()}
destination = Path(os.environ['LAB_RESULT_CATALOG']).resolve()
catalog.normalize_hrefs(str(destination))
for item in items:
    item_path = item.get_self_href()
    assert item_path is not None
    for key, asset in item.assets.items():
        assert Path(key).name == key and key not in {'', '.', '..'}
        source = asset_sources[(item.id, key)]
        assert source is not None
        parsed = urlparse(source)
        suffix = Path(parsed.path).suffix or '.bin'
        output = Path(item_path).parent / f'{key}{suffix}'
        output.parent.mkdir(parents=True, exist_ok=True)
        if parsed.scheme in {'http', 'https'}:
            command = ['curl', '--fail-with-body', '--location']
            if os.environ.get('LAB_CURL_CONFIG'):
                command.extend(['--config', os.environ['LAB_CURL_CONFIG']])
            subprocess.run([*command, '--output', str(output), source], check=True)
        elif parsed.scheme in {'', 'file'}:
            shutil.copyfile(unquote(parsed.path), output)
        else:
            raise ValueError('Use the backend staging adapter for this asset URI scheme')
        asset.href = str(output)
catalog.make_all_asset_hrefs_relative()
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
print('Complete result retrieved:', destination)
PYTHON
else
    printf 'Result retrieval skipped: no live job was requested.\n'
fi
)


In [ ]:
(
set -euo pipefail
if [[ "$LAB_RUN_DEPLOYMENT" == 1 ]]; then
    : "${LAB_RESULT_CATALOG:?Retrieve the complete remote product and set its local catalog directory}"
    python - <<'PYTHON'
import hashlib
import os
from pathlib import Path
import rasterio
from waterbodies.common import read_item
from waterbodies.validation import validate_item
item = read_item(Path(os.environ['LAB_RESULT_CATALOG']))
validate_item(item)
href = item.assets['data'].get_absolute_href()
assert href is not None
with rasterio.open(href) as raster:
    print('Remote mask:', raster.shape, raster.crs)
print('Remote STAC validation passed:', item.id)
Path('completion.txt').write_text('\n'.join([
    os.environ['EOAP_REF'], os.environ['PROCESS_ID'],
    'returned-package-sha256:' + hashlib.sha256(Path('installed-package.json').read_bytes()).hexdigest(),
    os.environ['JOB_ID'],
    os.environ['LAB_RESULT_CATALOG'], 'STAC validation passed']) + '\n')
PYTHON
else
    printf 'Remote result validation skipped: no live output was produced.\n'
fi
)


## Record your findings

**Expected outcome:** Local evidence checks pass and, with ORAS, the OCI round trip preserves the contract. Fresh scans and live deployment require their configured services; retain real digest, process, successful job and validated result evidence for those sections.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/13-secure-promote-deploy/solution.md) after the activity. Return to [the lab index](README.md).
